In [1]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from sklearn.datasets import load_digits
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC

SHOTS = 8192

In [2]:
from iqm.qiskit_iqm import transpile_to_IQM
from py4lexis.session import LexisSession
from qaas.client import QProvider, QBackend
from qaas import qiskit_to_pulla, sweep_job_to_qiskit

lexis_session = LexisSession()
token = lexis_session.get_access_token()

provider = QProvider("vlq_deic_dk_demo_project", token)
backend = provider.get_backend("VLQ-DK")
pulla = provider.get_pulla("VLQ-DK")
print("Successfully connected to VLQ!")

Welcome to the Py4LEXIS (v7.1.1)!
Proceeding login via login page...
Open provided url in your browser, please: https:/aai.lexis.tech/auth/realms/LEXIS_AAI_v2/protocol/openid-connect/auth/device?user_code=WRJH-UBIW
Check if user is logged in: Pending.                                            
Check if user is logged in: OK!
You have been successfully logged in LEXIS session.
Successfully connected to VLQ!


1. Dataset: 10 training and 10 testing images per digit

In [3]:
digits = load_digits()

rng_digits = np.random.default_rng(seed=0)

train_idx, test_idx = [], []
for d in range(10):
    idx_d = np.where(digits.target == d)[0]       # all images of digit d
    idx_d = rng_digits.permutation(idx_d)         # shuffle them
    train_idx += list(idx_d[:10])                 # first 10 -> train
    test_idx  += list(idx_d[10:20])               # next 10 -> test

X_train_raw = digits.data[train_idx]
y_train     = digits.target[train_idx]
X_test_raw  = digits.data[test_idx]
y_test      = digits.target[test_idx]

2. Compression to n features: truncated SVD + scaling to [0, 1]

In [4]:
def compression(X_train_raw, X_test_raw, n):
    svd = TruncatedSVD(n_components=n, random_state=0)
    svd.fit(X_train_raw)                         # learns the n directions, using only train
    Z_train = svd.transform(X_train_raw)
    Z_test  = svd.transform(X_test_raw)

    scaler = MinMaxScaler(feature_range=(0, 1))
    scaler.fit(Z_train)                          # learns min and max, using only train
    X_train = scaler.transform(Z_train)
    X_test  = np.clip(scaler.transform(Z_test), 0, 1)
    return X_train, X_test

3. Kernel circuit: U(x_j) followed by U(x_i)^dagger 

In [ ]:
def feature_map(x, n):          # n = number of qubits = number of features
    qc = QuantumCircuit(n)

    # Feature map, 4 layers (Sec. 3: "increase the depth to 4")
    for _ in range(4):
        for i in range(n):
            qc.h(i)
            qc.rz(2 * x[i], i)
        for i in range(n - 1):          # RZZ on neighbouring pairs (0,1), (1,2), ...
            x_ij = 2 * (np.pi - x[i]) * (np.pi - x[i + 1])
            qc.rzz(x_ij, i, i + 1)
    return qc


def kernel_circuit(x_i, x_j, n):
    qc = feature_map(x_j, n)                                 # U(x_j)
    qc.compose(feature_map(x_i, n).inverse(), inplace=True)  # U(x_i)^dagger
    qc.measure_all()
    return transpile_to_IQM(qc, backend)

4. Kernel matrices: K = frequency of the bitstring 0...0

In [ ]:
BATCH_SIZE = 50  

def run_on_vlq(circuits, shots):
    all_counts = []
    for b in range(0, len(circuits), BATCH_SIZE):
        batch = circuits[b:b + BATCH_SIZE]
        print(f"Enviando circuitos {b} a {b + len(batch) - 1} de {len(circuits)}")
        pulla_circuits, compiler = qiskit_to_pulla(pulla, backend, batch)
        settings = compiler.get_settings(circuits=pulla_circuits)
        settings.set_shots(shots)
        job_definition, context = compiler.compile(pulla_circuits, settings=settings)
        job = pulla.submit_playlist(job_definition, context=context)
        job.wait_for_completion()
        result = sweep_job_to_qiskit(job, shots=shots)
        all_counts += [result.get_counts(k) for k in range(len(batch))]
    return all_counts

def kernel_matrix_train(X_train, n, shots=SHOTS):
    N = len(X_train)
    zero = "0" * n                               # the all-zero bitstring

    # 1. Only the upper triangle (j > i)
    pairs = [(i, j) for i in range(N) for j in range(i + 1, N)]
    circuits = [kernel_circuit(X_train[i], X_train[j], n) for i, j in pairs]

    # 2. Run all circuits at once
    results = run_on_vlq(circuits, shots=shots)

    # 3. Diagonal = 1, fill each cell and its mirror
    K = np.eye(N)
    for (i, j), r in zip(pairs, results):
        counts = r.data.meas.get_counts()
        K[i, j] = counts.get(zero, 0) / shots
        K[j, i] = K[i, j]                        # symmetric
    return K


def kernel_matrix_test(X_test, X_train, n, shots=SHOTS):
    zero = "0" * n

    # 1. All (test, train) pairs
    pairs = [(i, j) for i in range(len(X_test)) for j in range(len(X_train))]
    circuits = [kernel_circuit(X_test[i], X_train[j], n) for i, j in pairs]

    # 2. Run all circuits at once
    results = run_on_vlq(circuits, shots=shots)

    # 3. Put each K in its cell
    K = np.zeros((len(X_test), len(X_train)))
    for (i, j), r in zip(pairs, results):
        counts = r.data.meas.get_counts()
        K[i, j] = counts.get(zero, 0) / shots
    return K

6. General case: n = 2, ..., 9

In [ ]:
def case2_gen(n):
    X_train, X_test = compression(X_train_raw, X_test_raw, n)

    K_train = kernel_matrix_train(X_train, n)
    K_test  = kernel_matrix_test(X_test, X_train, n)

    svm = SVC(kernel="precomputed")
    svm.fit(K_train, y_train)
    acc = svm.score(K_test, y_test)

    print(f"n = {n}: test accuracy = {acc:.2f}")
    return acc, K_train, K_test

7. Recreate Fig. 4(a) and 4(d)

In [ ]:
n_list = [2, 3, 4, 5, 6, 7, 8, 9]    # number of qubits

accs, K_trains = [], {}
for n in n_list:
    acc, K_train, K_test = case2_gen(n)
    accs.append(acc)
    K_trains[n] = K_train
    K_tests[n] = K_test
    np.save("results_case2_vlq.npy",
            {"n_list": n_list, "accs": accs, "K_trains": K_trains,
             "K_tests": K_tests, "y_train": y_train, "y_test": y_test},
            allow_pickle=True)
# Fig. 4(a)
plt.plot(n_list, accs, "o-", label="Standard")
plt.xlabel("Number of qubits"); plt.ylabel("Classification accuracy")
plt.xticks(n_list); plt.legend()
plt.show()

# Fig. 4(d)
plt.imshow(K_trains[9], cmap="gray_r", vmin=0, vmax=1)
plt.colorbar()
plt.title("Training kernel matrix, n = 9")
plt.savefig("fig4d.jpg", bbox_inches="tight")
plt.show()

n = 2: test accuracy = 0.29


n = 3: test accuracy = 0.40


n = 4: test accuracy = 0.55


n = 5: test accuracy = 0.62
